A4. D-study: обоснование дизайна «1 прогон × 229 кейсов» (Messing 2026, Shyr 2025).....


In [1]:
import sys, os

def _build_data_tools():
    import json
    import os
    import re
    import warnings
    import numpy as np
    import pandas as pd
    from scipy import stats
    warnings.filterwarnings('ignore')

    def path_join(*parts):
        return os.path.normpath(os.path.join(*parts))

    def normalize_newlines(frame):
        for column in frame.select_dtypes(include=['object', 'string']).columns:
            frame[column] = frame[column].map(lambda value: value.replace('\r\n', '\n').replace('\r', '\n') if isinstance(value, str) else value)
        return frame
    from pathlib import Path
    _start = Path.cwd().resolve()
    _repo = next((p for p in (_start, *_start.parents) if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()), None)
    if _repo is None:
        raise FileNotFoundError('Run the notebook from inside the repository.')
    if os.name == 'nt' and (not str(_repo).startswith(chr(92) * 2 + '?' + chr(92))):
        _repo = Path(chr(92) * 2 + '?' + chr(92) + str(_repo))
    ROOT = str(_repo)
    HERE = str(_repo / 'analyses/statistical_inference')
    OUT = path_join(HERE, 'results')
    os.makedirs(OUT, exist_ok=True)
    PBASE_DIR = 'experiments/clinical_protocols/delphi_panels/heterogeneous_gpt120_organizer_complete'
    HOMOG_DS_DIR = 'experiments/homogeneous_panels/three_deepseekv4_members_complete'
    NB = 'experiments/clinical_protocols/single_model_baselines/baseline_%s/llm_as_judge/baseline_%s_%s_judge.csv'
    USC = 'experiments/token_budget_controls/universal_self_consistency_gpt_seven_samples/llm_as_judge/usc_gpt_sc%d_judge.csv'
    HOMOG_DIRS = {'gpt': 'experiments/homogeneous_panels/three_gpt_members', 'qwen': 'experiments/homogeneous_panels/three_qwen_members', 'gemini': 'experiments/homogeneous_panels/three_gemini_members'}
    PANELS_OTHER_DIR = {'P-weak': 'experiments/clinical_protocols/delphi_panels/panel_qwen_organizer_gemini_deepseek_pro', 'P-mixed': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_gemini_glm', 'P-strong': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_glm_gpt120'}
    PANEL_FILE = {'P-weak': 'delphi_pweak_noroles_judge.csv', 'P-mixed': 'delphi_pmixed_noroles_judge.csv', 'P-strong': 'delphi_pstrong_noroles_judge.csv'}
    SINGLES = ['gpt', 'qwen', 'deepseekv4', 'deepseekv4pro', 'gemini', 'glm', 'kimi']
    COMPOSITION = {'P-base': ['gpt', 'qwen', 'deepseekv4'], 'P-weak': ['qwen', 'gemini', 'deepseekv4pro'], 'P-mixed': ['deepseekv4', 'gemini', 'glm'], 'P-strong': ['deepseekv4', 'glm', 'gpt'], 'homog gpt': ['gpt'] * 3, 'homog qwen': ['qwen'] * 3, 'homog deepseekv4': ['deepseekv4'] * 3, 'homog gemini': ['gemini'] * 3}
    HETERO = {'P-base': 1, 'P-weak': 1, 'P-mixed': 1, 'P-strong': 1, 'homog gpt': 0, 'homog qwen': 0, 'homog deepseekv4': 0, 'homog gemini': 0}
    PANELS = list(COMPOSITION)
    LADDER = ['baseline gpt', 'USC@3 gpt', 'USC@5 gpt', 'USC@7 gpt', 'homog gpt', 'homog deepseekv4', 'P-base']

    def arm_paths(design='noroles'):
        p = {}
        for m in SINGLES:
            p['baseline ' + m] = NB % (m, design, m)
        if design == 'noroles':
            for k in (3, 5, 7):
                p['USC@%d gpt' % k] = USC % k
            for tag, d in HOMOG_DIRS.items():
                p['homog ' + tag] = d + '/llm_as_judge/delphi_homog_%s_noroles_judge.csv' % tag
            p['homog deepseekv4'] = HOMOG_DS_DIR + '/llm_as_judge/delphi_homog_deepseekv4_noroles_judge.csv'
            for k, d in PANELS_OTHER_DIR.items():
                p[k] = d + '/llm_as_judge/' + PANEL_FILE[k]
        p['P-base'] = PBASE_DIR + '/llm_as_judge/delphi_base_%s_judge.csv' % design
        return p
    ROUNDS_DIR = {'P-base': (PBASE_DIR, 'delphi_base_noroles_protocol_r%d_judge.csv'), 'P-weak': (PANELS_OTHER_DIR['P-weak'], 'delphi_pweak_noroles_protocol_r%d_judge.csv'), 'P-mixed': (PANELS_OTHER_DIR['P-mixed'], 'delphi_pmixed_noroles_protocol_r%d_judge.csv'), 'P-strong': (PANELS_OTHER_DIR['P-strong'], 'delphi_pstrong_noroles_protocol_r%d_judge.csv'), 'homog gpt': (HOMOG_DIRS['gpt'], 'delphi_homog_gpt_noroles_protocol_r%d_judge.csv'), 'homog gemini': (HOMOG_DIRS['gemini'], 'delphi_homog_gemini_noroles_protocol_r%d_judge.csv'), 'homog deepseekv4': (HOMOG_DS_DIR, 'delphi_homog_deepseekv4_noroles_protocol_r%d_judge.csv')}
    RETEST = {'P-base': PBASE_DIR + '/llm_as_judge/judge_robustness/delphi_base_%s_retest_r%d_judge.csv', 'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_robustness/baseline_%s_gpt_retest_r%d_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_robustness/baseline_%s_qwen_retest_r%d_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_robustness/baseline_%s_deepseekv4_retest_r%d_judge.csv'}
    SHORT = {'P-base': PBASE_DIR + '/llm_as_judge/judge_robustness/delphi_base_%s_short_judge.csv', 'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_robustness/baseline_%s_gpt_short_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_robustness/baseline_%s_qwen_short_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_robustness/baseline_%s_deepseekv4_short_judge.csv'}
    JUDGE41 = {'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_gpt41/baseline_%s_gpt_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_gpt41/baseline_%s_qwen_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_gpt41/baseline_%s_deepseekv4_judge.csv'}
    CASE_COVARS = ['Type', 'Sub Clinical Specialties', 'Age Group', 'Genealogy', 'Diagnosis10', 'Vignette_race']
    DEXP = 'experiments/panel_parameter_sweeps'
    SWEEPS = {'experts': (DEXP + '/experts/llm_as_judge', 'delphi_experts_%s_judge.csv'), 'questionnaire_items': (DEXP + '/questionnaire_items/llm_as_judge', 'delphi_questionnaire_items_%s_judge.csv'), 'questions_per_expert': (DEXP + '/questions_per_expert/llm_as_judge', 'delphi_questions_per_expert_%s_judge.csv'), 'temperature': (DEXP + '/temperature/llm_as_judge', 'delphi_temperature_%s_judge.csv'), 'rounds': (DEXP + '/rounds/llm_as_judge', 'delphi_%s_judge.csv')}
    EXPERT_GROUPS = {'group1_10models': 10, 'group2_4models': 4, 'group3_5models': 5, 'group4_5models': 5, 'group5_7models': 7, 'group6_5models': 5, 'group7_6models': 6}
    EXT = 'experiments'
    MEDMCQA = {'baseline deepseekv4': EXT + '/external_benchmarks/medmcqa/baseline_deepseekv4/baseline_medmcqa_deepseekv4.csv', 'baseline gpt': EXT + '/external_benchmarks/medmcqa/baseline_gpt/baseline_medmcqa_gpt.csv', 'baseline qwen': EXT + '/external_benchmarks/medmcqa/baseline_qwen/baseline_medmcqa_qwen.csv', 'baseline alice': EXT + '/external_benchmarks/initial_runs/baseline_alice/baseline_medmcqa_alice.csv', 'baseline qwen36': EXT + '/external_benchmarks/initial_runs/baseline_qwen36/baseline_medmcqa_qwen36.csv', 'P-ac (gpt/qwen/ds)': EXT + '/external_benchmarks/medmcqa/panel_gpt120_organizer_qwen_deepseek/delphi_medmcqa_noroles.csv', 'P-ac (gpt/qwen/alice)': EXT + '/external_benchmarks/initial_runs/panel_gpt120_organizer_qwen_alice/delphi_medmcqa_noroles.csv', 'P-deepseektrio': EXT + '/external_benchmarks/initial_runs/three_deepseek_members/delphi_medmcqa_noroles.csv', 'P-base-like': EXT + '/external_benchmarks/initial_runs/heterogeneous_gpt120_organizer/delphi_medmcqa_noroles.csv'}

    def rd_any(rel, index=None):
        p = path_join(ROOT, rel)
        if not os.path.exists(p):
            return None
        with open(p, encoding='utf-8-sig', errors='replace') as fh:
            head = fh.readline()
        sep = ';' if head.count(';') > head.count(',') else ','
        d = normalize_newlines(pd.read_csv(p, sep=sep, encoding='utf-8-sig', low_memory=False))
        if 'table_source' in d.columns:
            d = d[d['table_source'].fillna('result_df') == 'result_df']
        if index and index in d.columns:
            d = d.dropna(subset=[index]).drop_duplicates(index).set_index(index)
        return d

    def sweep_arms(kind, verbose=True):
        import glob as _g
        folder, _ = SWEEPS[kind]
        out = {}
        for p in sorted(_g.glob(path_join(ROOT, folder, '*_judge.csv'))):
            name = os.path.basename(p)[:-len('_judge.csv')]
            d = rd(os.path.relpath(p, ROOT).replace('\\', '/'))
            if d is not None:
                out[name] = d
        if verbose:
            print('свип %s: плеч %d, кейсов %d' % (kind, len(out), len(next(iter(out.values()))) if out else 0))
        return out

    def member_triage_acc(d):
        t = true_label(d)
        out = {}
        for p in participants(d):
            lab = d['r1_emergency__' + p].map(norm_label)
            ok = lab.notna() & t.notna()
            out[p] = float((lab[ok] == t[ok]).mean()) if ok.any() else np.nan
        return pd.Series(out)

    def rd(rel, need_score=True):
        p = path_join(ROOT, rel)
        if not os.path.exists(p):
            return None
        d = normalize_newlines(pd.read_csv(p, sep=';', encoding='utf-8-sig', low_memory=False))
        if 'table_source' in d.columns:
            d = d[d['table_source'].fillna('result_df') == 'result_df']
        if need_score and 'Judge_score' not in d.columns:
            return None
        sub = ['Judge_score', 'Vignette'] if need_score else ['Vignette']
        d = d.dropna(subset=sub).drop_duplicates('Vignette', keep='first')
        return d.set_index('Vignette')

    def load_arms(design='noroles', verbose=True):
        a = {}
        for name, rel in arm_paths(design).items():
            d = rd(rel)
            if d is not None:
                a[name] = d
        if verbose:
            print('плеч загружено: %d' % len(a))
            print('   ' + ', '.join(a))
        return a

    def js(arms, name):
        return pd.to_numeric(arms[name]['Judge_score'], errors='coerce').dropna()

    def score_matrix(arms, names=None):
        names = list(arms) if names is None else names
        m = pd.concat([js(arms, n).rename(n) for n in names], axis=1)
        return m.dropna()

    def meta(arms=None):
        d = rd(PBASE_DIR + '/llm_as_judge/delphi_base_noroles_judge.csv')
        cols = [c for c in CASE_COVARS if c in d.columns]
        return d[cols].copy()

    def item_frame(d):
        r = {}
        if d is None or 'judge_details' not in d.columns:
            return r
        for v, s in d['judge_details'].dropna().items():
            try:
                arr = json.loads(s)
            except Exception:
                continue
            for it in arr:
                ax = next((t.split(':', 1)[1] for t in it.get('tags') or [] if isinstance(t, str) and t.startswith('axis:')), None)
                r[v, it['id']] = {'met': bool(it.get('met')), 'points': it.get('points', 0) or 0, 'axis': ax}
        return r

    def load_items(design='noroles', arms_wanted=None, verbose=True):
        paths = arm_paths(design)
        if arms_wanted:
            paths = {k: v for k, v in paths.items() if k in arms_wanted}
        items = {}
        for name, rel in paths.items():
            d = rd(rel)
            f = item_frame(d)
            if f:
                items[name] = f
        common = None
        for f in items.values():
            common = set(f) if common is None else common & set(f)
        ref = next(iter(items.values()))
        pos = sorted([k for k in common or set() if ref[k]['points'] > 0])
        if verbose:
            print('плеч с попунктными вердиктами: %d' % len(items))
            print('общих положительных пунктов: %d' % len(pos))
        return (items, pos)

    def items_long(items, pos, arms_wanted=None):
        names = arms_wanted or list(items)
        rows = []
        for a in names:
            f = items[a]
            for k in pos:
                it = f[k]
                rows.append({'case': k[0], 'cid': k[1], 'arm': a, 'met': int(it['met']), 'points': it['points'], 'axis': it['axis']})
        return pd.DataFrame(rows)

    def paired(a, b):
        ix = a.index.intersection(b.index)
        x, y = (a.loc[ix].astype(float), b.loc[ix].astype(float))
        d = x - y
        p = stats.wilcoxon(x, y)[1] if (d != 0).any() else np.nan
        t = stats.ttest_rel(x, y)
        return {'n': len(ix), 'delta': float(d.mean()), 'sd': float(d.std(ddof=1)), 'share_gt': float((d > 0).mean()), 'p_wilcoxon': float(p), 't': float(t.statistic), 'p_ttest': float(t.pvalue)}

    def se_clt(x):
        x = np.asarray(x, float)
        n = len(x)
        return float(np.sqrt(x.var(ddof=1) / n))

    def se_clustered(x, cluster, cr1=True):
        x = np.asarray(x, float)
        cl = np.asarray(cluster)
        n = len(x)
        e = x - x.mean()
        df = pd.DataFrame({'e': e, 'c': cl})
        r = df.groupby('c', observed=True)['e'].sum().to_numpy()
        ss = df.groupby('c', observed=True)['e'].apply(lambda s: float((s ** 2).sum())).to_numpy()
        cross = float((r ** 2 - ss).sum())
        v = se_clt(x) ** 2 + cross / n ** 2
        G = len(r)
        if cr1 and G > 1:
            v *= G / (G - 1.0)
        return (float(np.sqrt(max(v, 0.0))), G)

    def cluster_test(d, cluster, cr1=True):
        d = pd.Series(d).astype(float)
        cl = pd.Series(cluster).reindex(d.index)
        ok = cl.notna()
        d, cl = (d[ok], cl[ok])
        m = float(d.mean())
        s0 = se_clt(d)
        s1, G = se_clustered(d, cl, cr1=cr1)
        dfree = max(G - 1, 1)
        t = m / s1 if s1 > 0 else np.nan
        p = float(2 * stats.t.sf(abs(t), dfree)) if np.isfinite(t) else np.nan
        tc = stats.t.ppf(0.975, dfree)
        return {'n': len(d), 'G': G, 'mean': m, 'se_naive': s0, 'se_cluster': s1, 'DEFF': (s1 / s0) ** 2 if s0 > 0 else np.nan, 't': float(t), 'df': dfree, 'p': p, 'ci_lo': m - tc * s1, 'ci_hi': m + tc * s1, 'p_naive': float(2 * stats.norm.sf(abs(m / s0))) if s0 > 0 else np.nan}

    def icc_anova(values, groups):
        df = pd.DataFrame({'y': np.asarray(values, float), 'g': np.asarray(groups)})
        df = df.dropna()
        grp = df.groupby('g', observed=True)['y']
        ni = grp.size().to_numpy().astype(float)
        k = len(ni)
        N = ni.sum()
        if k < 2 or N <= k:
            return (0.0, float(ni.mean() if k else 1), k)
        gm = df['y'].mean()
        msb = float((ni * (grp.mean().to_numpy() - gm) ** 2).sum() / (k - 1))
        ssw = float(((df['y'] - df['g'].map(grp.mean()).to_numpy()) ** 2).sum())
        msw = ssw / (N - k)
        B0 = float((N - (ni ** 2).sum() / N) / (k - 1))
        rho = (msb - msw) / (msb + (B0 - 1) * msw) if msb + (B0 - 1) * msw > 0 else 0.0
        return (float(np.clip(rho, 0, 1)), B0, k)

    def ci_wald(p, n, z=1.96):
        se = np.sqrt(p * (1 - p) / n)
        return (p - z * se, p + z * se)

    def ci_wilson(p, n, z=1.96):
        d = 1 + z ** 2 / n
        c = (p + z ** 2 / (2 * n)) / d
        h = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / d
        return (c - h, c + h)

    def ci_agresti_coull(x, n, z=1.96):
        nt = n + z ** 2
        pt = (x + z ** 2 / 2) / nt
        se = np.sqrt(pt * (1 - pt) / nt)
        return (pt - z * se, pt + z * se)

    def ci_clopper_pearson(x, n, alpha=0.05):
        lo = 0.0 if x == 0 else stats.beta.ppf(alpha / 2, x, n - x + 1)
        hi = 1.0 if x == n else stats.beta.ppf(1 - alpha / 2, x + 1, n - x)
        return (float(lo), float(hi))

    def boot_ci(x, stat=np.mean, B=5000, seed=0, alpha=0.05):
        rng = np.random.default_rng(seed)
        x = np.asarray(x, float)
        n = len(x)
        s = np.array([stat(x[rng.integers(0, n, n)]) for _ in range(B)])
        return (float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2)))

    def boot_ci_cluster(x, cluster, stat=np.mean, B=5000, seed=0, alpha=0.05):
        rng = np.random.default_rng(seed)
        df = pd.DataFrame({'x': np.asarray(x, float), 'c': np.asarray(cluster)})
        groups = [g['x'].to_numpy() for _, g in df.groupby('c', observed=True)]
        G = len(groups)
        s = []
        for _ in range(B):
            pick = rng.integers(0, G, G)
            s.append(stat(np.concatenate([groups[i] for i in pick])))
        s = np.array(s)
        return (float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2)))

    def holm(pvals):
        p = np.asarray(pvals, float)
        m = len(p)
        order = np.argsort(p)
        adj = np.empty(m)
        run = 0.0
        for i, k in enumerate(order):
            run = max(run, (m - i) * p[k])
            adj[k] = min(run, 1.0)
        return adj

    def bh(pvals):
        p = np.asarray(pvals, float)
        m = len(p)
        order = np.argsort(p)
        adj = np.empty(m)
        run = 1.0
        for i in range(m - 1, -1, -1):
            k = order[i]
            run = min(run, m * p[k] / (i + 1))
            adj[k] = min(run, 1.0)
        return adj

    def bonferroni(pvals):
        p = np.asarray(pvals, float)
        return np.minimum(p * len(p), 1.0)

    def mcnemar(a, b):
        ix = a.index.intersection(b.index)
        x, y = (a.loc[ix].astype(bool), b.loc[ix].astype(bool))
        n01, n10 = (int((~x & y).sum()), int((x & ~y).sum()))
        if n01 + n10 == 0:
            return {'n01': n01, 'n10': n10, 'p': 1.0}
        return {'n01': n01, 'n10': n10, 'p': float(stats.binomtest(n10, n01 + n10, 0.5).pvalue)}
    ORD = {'SELF': 0, 'NONEMERG': 1, 'EMERG': 2}
    GT = {'Self-care appropriate': 'SELF', 'Requires non-emergent care': 'NONEMERG', 'Requires emergent care': 'EMERG'}
    _DASH = dict.fromkeys(map(ord, '‐‑‒–—―−\xad'), '-')
    _KEYS = [('requires emergent care', 'EMERG'), ('requires non-emergent care', 'NONEMERG'), ('self-care appropriate', 'SELF')]

    def norm_label(x):
        if not isinstance(x, str) or not x.strip():
            return None
        if x.strip() in GT:
            return GT[x.strip()]
        import unicodedata
        s = unicodedata.normalize('NFKC', x).translate(_DASH).lower()
        s = re.sub('\\s+', ' ', re.sub('[^a-z\\- ]', ' ', s)).strip()
        for k, v in _KEYS:
            if s.startswith(k) or k in s:
                return v
        if 'non-emergent' in s or 'non emergent' in s:
            return 'NONEMERG'
        if 'emergent' in s or 'emergency' in s:
            return 'EMERG'
        if 'self' in s:
            return 'SELF'
        return None
    _PAT = re.compile('emergency\\s*type\\s*[:\\-]\\s*[\\*_`#\\s]*([^\\n\\r\\*_|]{0,90})', re.I)

    def parse_triage(text):
        if not isinstance(text, str) or not text.strip():
            return None
        import unicodedata
        m = _PAT.search(unicodedata.normalize('NFKC', text).translate(_DASH))
        if not m:
            return None
        return norm_label('Requires ' + m.group(1) if False else m.group(1))

    def pred_triage(d):
        if 'delphi_consensus_emergency_type' in d.columns:
            s = d['delphi_consensus_emergency_type'].map(norm_label)
            if s.notna().mean() > 0.5:
                return s
        for c in ('winner_protocol', 'baseline_protocol_anonymous'):
            if c in d.columns:
                s = d[c].map(parse_triage)
                if s.notna().mean() > 0.3:
                    return s
        return pd.Series(index=d.index, dtype=object)

    def participants(d):
        return sorted((c.split('__', 1)[1] for c in d.columns if c.startswith('r1_emergency__')))

    def true_label(d):
        return d['Type'].map(GT)

    def words(s):
        return len(re.findall('\\S+', str(s))) if isinstance(s, str) else np.nan

    def fmt_p(p):
        if p is None or (isinstance(p, float) and (not np.isfinite(p))):
            return '  n/a '
        return '%.3f' % p if p >= 0.001 else '%.1e' % p

    def show(df, title=None, floatfmt='%.4f'):
        if title:
            print('\n' + title)
            print('-' * max(len(title), 60))
        with pd.option_context('display.width', 200, 'display.max_columns', 60, 'display.float_format', lambda v: floatfmt % v):
            print(df)

    def save(df, name):
        p = path_join(OUT, name)
        df.to_csv(p, sep=';', encoding='utf-8-sig')
        print('сохранено: out/%s  (%d строк)' % (name, len(df)))
        return p
    return locals()
globals().update({k: v for k, v in _build_data_tools().items() if not k.startswith('_')})
del _build_data_tools
A = load_arms()
M = meta()


плеч загружено: 18
   baseline gpt, baseline qwen, baseline deepseekv4, baseline deepseekv4pro, baseline gemini, baseline glm, baseline kimi, USC@3 gpt, USC@5 gpt, USC@7 gpt, homog gpt, homog qwen, homog gemini, homog deepseekv4, P-weak, P-mixed, P-strong, P-base


In [2]:
def series(rel):
    d = rd(rel)
    return None if d is None else pd.to_numeric(d['Judge_score'], errors='coerce').dropna()

def var_components(runs):
    ix = sorted(set.intersection(*[set(s.index) for s in runs.values()]))
    Y = np.column_stack([runs[k].loc[ix].to_numpy(float) for k in sorted(runs)])
    n, R = Y.shape
    gm = Y.mean()
    ms_case = R * ((Y.mean(axis=1) - gm) ** 2).sum() / (n - 1)
    ms_run = n * ((Y.mean(axis=0) - gm) ** 2).sum() / (R - 1)
    resid = Y - Y.mean(axis=1, keepdims=True) - Y.mean(axis=0, keepdims=True) + gm
    ms_err = (resid ** 2).sum() / ((n - 1) * (R - 1))
    return {'n': n, 'R': R, 'v_case': max((ms_case - ms_err) / R, 0.0), 'v_run': max((ms_run - ms_err) / n, 0.0), 'v_resid': ms_err}
GS = {}
for arm, tpl in RETEST.items():
    runs = {}
    for r in range(1, 6):
        s = series(tpl % ('noroles', r))
        if s is not None:
            runs['r%d' % r] = s
    if len(runs) >= 2:
        GS[arm] = var_components(runs)
gs = pd.DataFrame(GS).T
gs['sd_case'] = np.sqrt(gs['v_case'])
gs['sd_run'] = np.sqrt(gs['v_run'])
gs['sd_resid'] = np.sqrt(gs['v_resid'])
show(gs[['n', 'R', 'sd_case', 'sd_run', 'sd_resid']], 'G-study: грань «прогон» (5 перезапусков x 60 кейсов)')



G-study: грань «прогон» (5 перезапусков x 60 кейсов)
------------------------------------------------------------
                          n      R  sd_case  sd_run  sd_resid
P-base              60.0000 5.0000   0.0844  0.0000    0.0869
baseline gpt        60.0000 5.0000   0.1441  0.0000    0.0856
baseline qwen       60.0000 5.0000   0.1264  0.0000    0.0929
baseline deepseekv4 60.0000 5.0000   0.1303  0.0014    0.0882


In [3]:
rows = []
for arm, tpl in SHORT.items():
    a_main = js(A, arm) if arm in A else None
    a_short = series(tpl % 'noroles')
    if a_main is None or a_short is None:
        continue
    ix = a_main.index.intersection(a_short.index)
    d = (a_main.loc[ix] - a_short.loc[ix]).astype(float)
    rows.append({'плечо': arm, 'n': len(ix), 'обычный': float(a_main.loc[ix].mean()), 'короткий': float(a_short.loc[ix].mean()), 'Δ': float(d.mean()), 'SD разности': float(d.std(ddof=1)), 'r Пирсона': float(np.corrcoef(a_main.loc[ix], a_short.loc[ix])[0, 1]), 'p': float(stats.wilcoxon(d)[1])})
PR = pd.DataFrame(rows).set_index('плечо')
show(PR, 'Грань «формулировка»: обычный против короткого промпта')



Грань «формулировка»: обычный против короткого промпта
------------------------------------------------------------
                       n  обычный  короткий      Δ  SD разности  r Пирсона      p
плечо                                                                            
P-base               229   0.7899    0.7267 0.0632       0.1636     0.4272 0.0000
baseline gpt         229   0.6258    0.5429 0.0829       0.1807     0.5094 0.0000
baseline qwen        229   0.5622    0.4683 0.0940       0.1753     0.5307 0.0000
baseline deepseekv4  228   0.6401    0.5306 0.1094       0.1844     0.4869 0.0000


In [4]:
rows = []
for arm, tpl in JUDGE41.items():
    a_main = js(A, arm) if arm in A else None
    a_j41 = series(tpl % 'noroles')
    if a_main is None or a_j41 is None:
        continue
    ix = a_main.index.intersection(a_j41.index)
    x, y = (a_main.loc[ix].astype(float), a_j41.loc[ix].astype(float))
    d = x - y
    rows.append({'плечо': arm, 'n': len(ix), 'основной': float(x.mean()), 'gpt-4.1': float(y.mean()), 'Δ': float(d.mean()), 'SD разности': float(d.std(ddof=1)), 'r Пирсона': float(np.corrcoef(x, y)[0, 1]), 'p': float(stats.wilcoxon(d)[1])})
JD = pd.DataFrame(rows).set_index('плечо')
show(JD, 'Грань «судья»: основной против gpt-4.1')



Грань «судья»: основной против gpt-4.1
------------------------------------------------------------
                      n  основной  gpt-4.1      Δ  SD разности  r Пирсона      p
плечо                                                                           
baseline gpt         60    0.6485   0.6050 0.0436       0.0993     0.8027 0.0015
baseline qwen        60    0.5854   0.5287 0.0567       0.1022     0.8020 0.0002
baseline deepseekv4  59    0.6580   0.6519 0.0060       0.1010     0.7761 0.5770


In [5]:
Ar = load_arms('randomroles', verbose=False)
rows = []
for arm in ['P-base', 'baseline gpt', 'baseline qwen', 'baseline deepseekv4']:
    if arm not in A or arm not in Ar:
        continue
    x, y = (js(A, arm), js(Ar, arm))
    ix = x.index.intersection(y.index)
    d = (x.loc[ix] - y.loc[ix]).astype(float)
    rows.append({'плечо': arm, 'n': len(ix), 'noroles': float(x.loc[ix].mean()), 'randomroles': float(y.loc[ix].mean()), 'Δ': float(d.mean()), 'SD разности': float(d.std(ddof=1)), 'p': float(stats.wilcoxon(d)[1])})
DS = pd.DataFrame(rows).set_index('плечо')
show(DS, 'Грань «дизайн ролей»')



Грань «дизайн ролей»
------------------------------------------------------------
                       n  noroles  randomroles       Δ  SD разности      p
плечо                                                                     
P-base               229   0.7899       0.8133 -0.0234       0.1213 0.0028
baseline gpt         229   0.6258       0.6488 -0.0229       0.1469 0.0519
baseline qwen        229   0.5622       0.5952 -0.0329       0.1295 0.0001
baseline deepseekv4  228   0.6401       0.6603 -0.0203       0.1556 0.0130


In [6]:
rows = []
for arm in gs.index:
    if arm not in A:
        continue
    r = {'плечо': arm, 'SD кейсов / sqrt(229)': float(np.sqrt(gs.loc[arm, 'v_case'] / 229)), 'SD прогона (R=1)': float(gs.loc[arm, 'sd_run'])}
    r['SD формулировки'] = float(PR.loc[arm, 'Δ']) if arm in PR.index else np.nan
    r['SD судьи'] = float(JD.loc[arm, 'Δ']) if arm in JD.index else np.nan
    g = M['Sub Clinical Specialties'].reindex(js(A, arm).index)
    gm = js(A, arm).groupby(g).mean()
    r['SD домена / sqrt(20)'] = float(np.sqrt(gm.var(ddof=1) / gm.size))
    rows.append(r)
BUD = pd.DataFrame(rows).set_index('плечо')
show(BUD, 'Вклад граней в неопределённость отчётного числа (в единицах Judge_score)')
print()
print('Читается так: «SD формулировки» и «SD судьи» — это СДВИГ, который возникает')
print('при смене промпта или судьи. Это не случайный шум, а систематическое смещение:')
print('оно не уменьшается с ростом числа кейсов.')



Вклад граней в неопределённость отчётного числа (в единицах Judge_score)
------------------------------------------------------------------------
                     SD кейсов / sqrt(229)  SD прогона (R=1)  SD формулировки  SD судьи  SD домена / sqrt(20)
плечо                                                                                                        
P-base                              0.0056            0.0000           0.0632       NaN                0.0146
baseline gpt                        0.0095            0.0000           0.0829    0.0436                0.0160
baseline qwen                       0.0084            0.0000           0.0940    0.0567                0.0147
baseline deepseekv4                 0.0086            0.0014           0.1094    0.0060                0.0127

Читается так: «SD формулировки» и «SD судьи» — это СДВИГ, который возникает
при смене промпта или судьи. Это не случайный шум, а систематическое смещение:
оно не уменьшается с ростом числа кей

In [7]:
ARM = 'P-base'
vc, vr, ve = gs.loc[ARM, ['v_case', 'v_run', 'v_resid']]
grid = []
for n in [30, 60, 100, 150, 229, 400, 800]:
    for R in [1, 2, 3, 5]:
        grid.append({'кейсов n': n, 'прогонов R': R, 'SE среднего': float(np.sqrt(vc / n + vr / R + ve / (n * R)))})
GRID = pd.DataFrame(grid).pivot(index='кейсов n', columns='прогонов R', values='SE среднего')
show(GRID, 'D-study: SE среднего Judge_score для P-base')



D-study: SE среднего Judge_score для P-base
------------------------------------------------------------
прогонов R      1      2      3      5
кейсов n                              
30         0.0221 0.0191 0.0179 0.0170
60         0.0156 0.0135 0.0127 0.0120
100        0.0121 0.0104 0.0098 0.0093
150        0.0099 0.0085 0.0080 0.0076
229        0.0080 0.0069 0.0065 0.0061
400        0.0061 0.0052 0.0049 0.0046
800        0.0043 0.0037 0.0035 0.0033


In [8]:
PAIRS = [('P-base', 'baseline gpt'), ('P-base', 'homog gpt'), ('P-base', 'homog deepseekv4'), ('P-base', 'P-strong')]
rows = []
zc = stats.norm.ppf(0.975) + stats.norm.ppf(0.8)
for a_name, b_name in PAIRS:
    if a_name not in A or b_name not in A:
        continue
    x, y = (js(A, a_name), js(A, b_name))
    ix = x.index.intersection(y.index)
    d = (x.loc[ix] - y.loc[ix]).astype(float)
    sd = float(d.std(ddof=1))
    row = {'контраст': '%s − %s' % (a_name, b_name), 'наблюдённая Δ': float(d.mean()), 'SD разностей': sd}
    for n in [60, 100, 229, 400]:
        row['MDD при n=%d' % n] = zc * sd / np.sqrt(n)
    rows.append(row)
MDD = pd.DataFrame(rows).set_index('контраст')
show(MDD, 'Минимальная обнаружимая разница (мощность 80%, α = 0.05)')



Минимальная обнаружимая разница (мощность 80%, α = 0.05)
------------------------------------------------------------
                           наблюдённая Δ  SD разностей  MDD при n=60  MDD при n=100  MDD при n=229  MDD при n=400
контраст                                                                                                         
P-base − baseline gpt             0.1641        0.1560        0.0564         0.0437         0.0289         0.0219
P-base − homog gpt                0.0312        0.1437        0.0520         0.0403         0.0266         0.0201
P-base − homog deepseekv4         0.0089        0.1262        0.0456         0.0353         0.0234         0.0177
P-base − P-strong                -0.0162        0.1315        0.0476         0.0368         0.0244         0.0184


In [9]:
rows = []
for a_name, b_name in PAIRS:
    if a_name not in A or b_name not in A:
        continue
    x, y = (js(A, a_name), js(A, b_name))
    ix = x.index.intersection(y.index)
    sd = float((x.loc[ix] - y.loc[ix]).std(ddof=1))
    row = {'контраст': '%s − %s' % (a_name, b_name), 'SD разностей': sd}
    for target in [0.01, 0.02, 0.03, 0.05, 0.1]:
        row['n для Δ=%.2f' % target] = int(np.ceil((zc * sd / target) ** 2))
    rows.append(row)
NEED = pd.DataFrame(rows).set_index('контраст')
show(NEED, 'Требуемое число кейсов (мощность 80%)')



Требуемое число кейсов (мощность 80%)
------------------------------------------------------------
                           SD разностей  n для Δ=0.01  n для Δ=0.02  n для Δ=0.03  n для Δ=0.05  n для Δ=0.10
контраст                                                                                                     
P-base − baseline gpt            0.1560          1910           478           213            77            20
P-base − homog gpt               0.1437          1622           406           181            65            17
P-base − homog deepseekv4        0.1262          1250           313           139            50            13
P-base − P-strong                0.1315          1358           340           151            55            14


In [10]:
NULLS = [('P-base', 'homog deepseekv4'), ('P-base', 'P-mixed'), ('P-base', 'P-strong'), ('USC@7 gpt', 'baseline gpt'), ('USC@3 gpt', 'baseline gpt'), ('USC@5 gpt', 'baseline gpt')]
rows = []
for a_name, b_name in NULLS:
    if a_name not in A or b_name not in A:
        continue
    x, y = (js(A, a_name), js(A, b_name))
    ix = x.index.intersection(y.index)
    d = (x.loc[ix] - y.loc[ix]).astype(float)
    sd, n = (float(d.std(ddof=1)), len(d))
    se = sd / np.sqrt(n)
    mdd = zc * se
    rows.append({'контраст': '%s − %s' % (a_name, b_name), 'n': n, 'Δ': float(d.mean()), 'SE': se, 'p': float(stats.wilcoxon(d)[1]), 'MDD (80%)': mdd, 'CI низ': d.mean() - 1.96 * se, 'CI верх': d.mean() + 1.96 * se, 'вывод': 'эквивалентность в пределах ±%.3f' % mdd})
NP_ = pd.DataFrame(rows).set_index('контраст')
show(NP_, 'Ретроспективная мощность незначимых контрастов')



Ретроспективная мощность незначимых контрастов
------------------------------------------------------------
                             n       Δ     SE      p  MDD (80%)  CI низ  CI верх                              вывод
контраст                                                                                                           
P-base − homog deepseekv4  229  0.0089 0.0083 0.1634     0.0234 -0.0075   0.0252  эквивалентность в пределах ±0.023
P-base − P-mixed           229  0.0035 0.0100 0.8837     0.0279 -0.0160   0.0231  эквивалентность в пределах ±0.028
P-base − P-strong          229 -0.0162 0.0087 0.0845     0.0244 -0.0332   0.0009  эквивалентность в пределах ±0.024
USC@7 gpt − baseline gpt   229  0.0021 0.0081 0.6984     0.0228 -0.0138   0.0181  эквивалентность в пределах ±0.023
USC@3 gpt − baseline gpt   229 -0.0081 0.0080 0.3983     0.0223 -0.0237   0.0075  эквивалентность в пределах ±0.022
USC@5 gpt − baseline gpt   229  0.0025 0.0080 0.4197     0.0225 -0.0132   0.018

In [11]:
save(gs.reset_index(), 'generalizability_and_sample_size_gstudy_components.csv')
save(BUD.reset_index(), 'generalizability_and_sample_size_facet_budget.csv')
save(GRID.reset_index(), 'generalizability_and_sample_size_dstudy_grid.csv')
save(NEED.reset_index(), 'generalizability_and_sample_size_required_n.csv')
save(NP_.reset_index(), 'generalizability_and_sample_size_retrospective_power.csv')


сохранено: out/generalizability_and_sample_size_gstudy_components.csv  (4 строк)
сохранено: out/generalizability_and_sample_size_facet_budget.csv  (4 строк)
сохранено: out/generalizability_and_sample_size_dstudy_grid.csv  (7 строк)
сохранено: out/generalizability_and_sample_size_required_n.csv  (4 строк)
сохранено: out/generalizability_and_sample_size_retrospective_power.csv  (6 строк)


'\\\\?\\D:\\research_llmfmed\\research_llmfmed\\1design_alpha\\dataverse_gold\\github_repository\\analyses\\statistical_inference\\results\\generalizability_and_sample_size_retrospective_power.csv'